In [2]:
%pip install opencv

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement opencv (from versions: none)
ERROR: No matching distribution found for opencv


In [3]:
%pip install mediapipe

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 0.0/22.4 MB ? eta -:--:--
    --------------------------------------- 0.5/22.4 MB 2.5 MB/s eta 0:00:09
   -- ------------------------------------- 1.3/22.4 MB 3.8 MB/s eta 0:00:06
   ---- ----------------------------------- 2.6/22.4 MB 4.5 MB/s eta 0:00:05
   ------- -------------------------------- 3.9/22.4 MB 5.3 MB/s eta 0:00:04
   ---------- ----------------------------- 6.0/22.4 MB 6.0 MB/s eta 0:00:03
   ----------- ---------------------------- 6.6/22.4 MB 5.6 MB/s eta 0:00:03
   -------------- ------------------------- 7.9/22.4 MB 5.6 MB/s eta 0:00:03
   --------------- ------------------------ 8.7/22.4 MB 5.3 MB/s eta 0:00:03
   ---------------- ------

In [16]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [45]:
import math
import numpy as np


def calculate_angle(a, b, c):
    """Calculate angle ABC, in degrees, from 2D landmark coordinates."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom < 1e-6:
        return None

    cosine = np.dot(ba, bc) / denom
    cosine = np.clip(cosine, -1.0, 1.0)
    return math.degrees(math.acos(cosine))


class SquatTracker:
    def __init__(self, deep_angle=95, start_angle=150, stand_angle=160):
        self.deep_angle = deep_angle
        self.start_angle = start_angle
        self.stand_angle = stand_angle

        self.reps = 0
        self.good_reps = 0
        self.state = "standing"
        self.min_angle = 180

    def update(self, angle):
        if angle is None:
            return self.state, "Pose unclear"

        if self.state == "standing":
            if angle < self.start_angle:
                self.state = "squatting"
                self.min_angle = angle

        elif self.state == "squatting":
            self.min_angle = min(self.min_angle, angle)

            if angle > self.stand_angle:
                self.reps += 1

                if self.min_angle <= self.deep_angle:
                    self.good_reps += 1
                    feedback = "Good depth!"
                else:
                    feedback = "Go deeper!"

                self.state = "standing"
                return self.state, feedback

        if self.state == "standing":
            return self.state, "Stand tall"

        if self.min_angle <= self.deep_angle:
            return self.state, "Depth reached"

        return self.state, "Keep going lower"


tracker = SquatTracker()

In [46]:
def get_knee_angle(landmarks, ids):
    hip, knee, ankle = [landmarks[i] for i in ids]

    visibility = [
        hip.visibility,
        knee.visibility,
        ankle.visibility
    ]

    if min(visibility) < 0.5:
        return None, min(visibility)

    angle = calculate_angle(
        (hip.x, hip.y),
        (knee.x, knee.y),
        (ankle.x, ankle.y)
    )
    return angle, min(visibility)

In [47]:
model_path = 'pose_landmarker_lite.task'  # Make sure this matches your file name
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,  # Specifically optimized for camera tracking
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Capture Video
cap = cv2.VideoCapture(0)

In [48]:
CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12),
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
    (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
    (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31), (27, 31),
    (24, 26), (26, 28), (28, 30), (30, 32), (28, 32),
]

In [49]:
with vision.PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened(): 
        ret, frame = cap.read() 
        if not ret: 
            print("Video capture ended.") 
            break 

        # The Tasks API requires converting BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Convert OpenCV frame to MediaPipe MPImage object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        
        # Get frame timestamp in milliseconds (mandatory for RunningMode.VIDEO)
        frame_timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
        # Fallback timestamp calculation if properties fail to resolve
        if frame_timestamp_ms == 0:
            frame_timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)

        # Process the frame
        detection_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # Draw tracking landmarks if anything is found
        if detection_result.pose_landmarks:            
            # MediaPipe landmark indices:
            # Left hip/knee/ankle: 23, 25, 27
            # Right hip/knee/ankle: 24, 26, 28

            landmarks = detection_result.pose_landmarks[0]

            left_ids = (23, 25, 27)
            right_ids = (24, 26, 28)

            
            left_angle, left_vis = get_knee_angle(landmarks, left_ids)
            right_angle, right_vis = get_knee_angle(landmarks, right_ids)
            
            # Use the more visible side.
            if left_vis >= right_vis:
                knee_angle = left_angle
            else:
                knee_angle = right_angle

            state, feedback = tracker.update(knee_angle)

            # Display the rep counter and feedback.
            cv2.putText(
                frame,
                f"Reps: {tracker.reps}  Good: {tracker.good_reps}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"State: {state}",
                (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 200, 0),
                2
            )

            cv2.putText(
                frame,
                f"Angle: {knee_angle:.0f}" if knee_angle is not None
                else "Angle: unavailable",
                (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

            cv2.putText(
                frame,
                feedback,
                (20, 145),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2
            )


            h, w, _ = frame.shape

            for pose_landmarks in detection_result.pose_landmarks:
                pixel_landmarks = []
                
                # Convert normalized coordinates to pixel coordinates
                for lm in landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_landmarks.append((cx, cy))
                    # Draw keypoint circles
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                for a, b in CONNECTIONS:
                    if pixel_landmarks[a] is not None and pixel_landmarks[b] is not None:
                        cv2.line(
                            frame, pixel_landmarks[a], pixel_landmarks[b],
                            (0, 180, 255), 2
                        )
                # Core skeletal connections (mapping index relationships)
                """
                connections = [
                    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), # Upper body / Arms
                    (11, 23), (12, 24), (23, 24),                     # Torso
                    (23, 25), (24, 26), (25, 27), (26, 28)            # Legs
                ]
                
                # Draw skeletal lines
                for [start_idx, end_idx] in connections:
                    if start_idx < len(pixel_landmarks) and end_idx < len(pixel_landmarks):
                        cv2.line(frame, pixel_landmarks[start_idx], pixel_landmarks[end_idx], (0, 0, 255), 2)
                """
        # Optional: Resize the preview frame to fit your display bounds
        display_frame = cv2.resize(frame, (960, 540)) 
        cv2.imshow('Pose Estimation', display_frame) 

        # Break loop if 'q' is pressed 
        if cv2.waitKey(1) & 0xFF == ord('q'): 
            break

cap.release() 
cv2.destroyAllWindows()